<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Regularization: Ridge, Lasso & Elastic Net</h1></center>

Regularization prevents overfitting by adding a penalty to the loss function.

### Topics:
1. [Why Regularization?](#1)
2. [Ridge Regression (L2)](#2)
3. [Lasso Regression (L1)](#3)
4. [Elastic Net](#4)
5. [Coefficient Paths & Comparison](#5)

In [ ]:
import seaborn as sns
reg_colors = ['#1b4332', '#40916c', '#b7e4c7', '#ff9f1c', '#2d6a4f']
print("Regularization Colors:")
sns.palplot(sns.color_palette(reg_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.datasets import fetch_california_housing
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Why Regularization?</h1></center>

## The Overfitting Problem

A model that is **too complex** memorises training data but fails on new data.
Regularization adds a penalty term to the loss to shrink coefficients:

| Method | Loss Function |
|---|---|
| OLS | $\sum(y_i - \hat{y}_i)^2$ |
| Ridge (L2) | $\sum(y_i - \hat{y}_i)^2 + \lambda\sum\beta_j^2$ |
| Lasso (L1) | $\sum(y_i - \hat{y}_i)^2 + \lambda\sum|\beta_j|$ |
| Elastic Net | $\sum(y_i - \hat{y}_i)^2 + \lambda[\alpha\sum|\beta_j| + (1-\alpha)\sum\beta_j^2]$ |

**Key insight**: Ridge shrinks all coefficients toward zero. Lasso can zero them out entirely — acting as automatic feature selection.

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

In [ ]:
# Overfitting demo with polynomial regression
np.random.seed(42)
X_demo = np.sort(np.random.uniform(0, 1, 40))
y_demo = np.sin(2 * np.pi * X_demo) + np.random.normal(0, 0.2, 40)
X_fine = np.linspace(0, 1, 300)

degrees = [1, 3, 9, 15]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), dpi=100)

for ax, d in zip(axes, degrees):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression())
    m.fit(X_demo.reshape(-1, 1), y_demo)
    rmse = np.sqrt(mean_squared_error(y_demo, m.predict(X_demo.reshape(-1, 1))))
    ax.scatter(X_demo, y_demo, s=20, color="#40916c", alpha=0.7)
    ax.plot(X_fine, np.sin(2*np.pi*X_fine), "k--", lw=1, label="True")
    ax.plot(X_fine, np.clip(m.predict(X_fine.reshape(-1,1)), -3, 3),
            color="#ff9f1c", lw=2)
    ax.set_title(f"Degree {d}\nRMSE={rmse:.3f}")
    ax.set_ylim(-2.5, 2.5)

plt.suptitle("Polynomial Overfitting — High Degree Memorises Noise", fontweight="bold")
plt.tight_layout()
plt.show()

<center><h1 style = "background:#1b4332 ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

**California Housing** — 20,640 rows, 8 features. Features are standardised before fitting.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f"Shape: {df.shape}")
df.describe().T

<center><h1 style = "background:#40916c ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
plt.figure(figsize=(10, 7), dpi=100)
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="Greens", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

<center><h1 style = "background:#b7e4c7 ;color:black;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_train)
X_te_s = sc.transform(X_test)

print(f"Train: {X_tr_s.shape}  Test: {X_te_s.shape}")

<a id = "2"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Ridge Regression (L2)</h1></center>

## Theory

Ridge adds the **squared magnitude** of coefficients as a penalty:

$$\min_\beta \sum(y_i - \hat{y}_i)^2 + \lambda\sum_{j=1}^p \beta_j^2$$

- All coefficients are **shrunk toward zero** but never reach exactly zero
- Best when many features are relevant (small but non-zero effects)
- $\lambda$ (alpha in sklearn) controls regularisation strength

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

<center><h1 style = "background:#ff9f1c ;color:black;border:0;font-weight:bold">Ridge Model</h1></center>

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 4, 100), cv=10)
ridge_cv.fit(X_tr_s, y_train)
pred_ridge = ridge_cv.predict(X_te_s)

print(f"Best alpha (λ): {ridge_cv.alpha_:.4f}")
print(f"Train R²      : {ridge_cv.score(X_tr_s, y_train):.4f}")
print(f"Test  R²      : {ridge_cv.score(X_te_s, y_test):.4f}")

<center><h1 style = "background:#2d6a4f ;color:white;border:0;font-weight:bold">Evaluation — Ridge</h1></center>

In [ ]:
k = X_te_s.shape[1]; n = len(X_te_s)
r2  = r2_score(y_test, pred_ridge)
adj = 1 - (1-r2)*(n-1)/(n-k-1)
pd.DataFrame({"Metric": ["MSE","RMSE","MAE","R²","Adjusted R²"],
              "Score": [mean_squared_error(y_test, pred_ridge),
                        np.sqrt(mean_squared_error(y_test, pred_ridge)),
                        mean_absolute_error(y_test, pred_ridge),
                        r2, adj]}).round(4)

<a id = "3"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Lasso Regression (L1)</h1></center>

## Theory

Lasso uses the **absolute magnitude** of coefficients:

$$\min_\beta \sum(y_i - \hat{y}_i)^2 + \lambda\sum_{j=1}^p |\beta_j|$$

- Coefficients can become **exactly zero** → automatic feature selection
- Best when you suspect many irrelevant features

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

<center><h1 style = "background:#ff9f1c ;color:black;border:0;font-weight:bold">Lasso Model</h1></center>

In [ ]:
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=10, max_iter=10000, random_state=42)
lasso_cv.fit(X_tr_s, y_train)
pred_lasso = lasso_cv.predict(X_te_s)

print(f"Best alpha (λ) : {lasso_cv.alpha_:.6f}")
n_zero = (np.abs(lasso_cv.coef_) < 1e-6).sum()
print(f"Zeroed features: {n_zero} of {X.shape[1]}")
print(f"Train R²       : {lasso_cv.score(X_tr_s, y_train):.4f}")
print(f"Test  R²       : {lasso_cv.score(X_te_s, y_test):.4f}")

<center><h1 style = "background:#2d6a4f ;color:white;border:0;font-weight:bold">Evaluation — Lasso</h1></center>

In [ ]:
r2l  = r2_score(y_test, pred_lasso)
adjl = 1 - (1-r2l)*(n-1)/(n-k-1)
pd.DataFrame({"Metric": ["MSE","RMSE","MAE","R²","Adjusted R²"],
              "Score": [mean_squared_error(y_test, pred_lasso),
                        np.sqrt(mean_squared_error(y_test, pred_lasso)),
                        mean_absolute_error(y_test, pred_lasso),
                        r2l, adjl]}).round(4)

<a id = "4"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Elastic Net</h1></center>

## Theory

Elastic Net combines L1 and L2 penalties:

$$\min_\beta \sum(y_i-\hat{y}_i)^2 + \lambda\left[\alpha\sum|\beta_j| + (1-\alpha)\sum\beta_j^2\right]$$

- `l1_ratio = 1` → Lasso; `l1_ratio = 0` → Ridge
- Best for correlated features where some selection is also desired

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

<center><h1 style = "background:#ff9f1c ;color:black;border:0;font-weight:bold">Elastic Net Model</h1></center>

In [ ]:
enet_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                       cv=10, max_iter=10000, random_state=42)
enet_cv.fit(X_tr_s, y_train)
pred_enet = enet_cv.predict(X_te_s)

print(f"Best alpha   : {enet_cv.alpha_:.4f}")
print(f"Best l1_ratio: {enet_cv.l1_ratio_:.2f}")
print(f"Train R²     : {enet_cv.score(X_tr_s, y_train):.4f}")
print(f"Test  R²     : {enet_cv.score(X_te_s, y_test):.4f}")

<a id = "5"></a>
<center><h1 style = "background:#2d6a4f ;color:white;border:0;font-weight:bold">Model Comparison</h1></center>

In [ ]:
ols = LinearRegression().fit(X_tr_s, y_train)
pred_ols = ols.predict(X_te_s)

models_res = {}
for name, pred_m in [("OLS", pred_ols), ("Ridge CV", pred_ridge),
                      ("Lasso CV", pred_lasso), ("ElasticNet", pred_enet)]:
    r2m = r2_score(y_test, pred_m)
    models_res[name] = {"RMSE": np.sqrt(mean_squared_error(y_test, pred_m)),
                         "MAE":  mean_absolute_error(y_test, pred_m),
                         "R²":   r2m}

comp_df = pd.DataFrame(models_res).T.round(4)
print(comp_df.to_string())

In [ ]:
# Coefficient paths
alphas = np.logspace(-3, 3, 150)
ridge_coefs = [Ridge(alpha=a).fit(X_tr_s, y_train).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_tr_s, y_train).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
for i, name in enumerate(X.columns):
    axes[0].plot(np.log10(alphas), [c[i] for c in ridge_coefs], label=name)
    axes[1].plot(np.log10(alphas), [c[i] for c in lasso_coefs], label=name)

for ax, title in zip(axes, ["Ridge Coefficient Paths", "Lasso Coefficient Paths"]):
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("log10(alpha)")
    ax.set_ylabel("Coefficient")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()